# Wav2Vec2 Fine-Tuning - Tshivenda (ven)

**Superseded for actual runs** - kept as the original reference version. The pilot-script + Colab-notebook pattern (`pilot_finetune_mps.py` / `colab_stage1_wav2vec2_ven.ipynb`) is what real runs use now; this notebook is where `colab_stage1_wav2vec2_ven.ipynb` ported its Stage 1 training-cell logic from, kept for reference.

Structure follows `finetune_wav2vec2.ipynb` (Khotso's Setswana smoke-test version),
adapted for the Tshivenda sub-study:
- Loads our preprocessed CSVs (`dataset/processed/`) instead of streaming from the Hub.
- Uses the shared committed tokenizer (`tokenizers/ven/`) so WER is comparable across runs.
- Transcripts are already normalised by preprocessing - no CHARS_TO_IGNORE cleaning here.
- Two-stage training per the proposal, selected via the `STAGE` config below:
  **Stage 1** = NCHLT + ANV combined; **Stage 2** = NCHLT-only, resuming from the Stage 1 model.

**Requires a CUDA GPU** (Colab or the team GPU machine). Everything up to section 6
runs on CPU for verification; the train cell will be very slow without a GPU.

**Kernel**: MultilingualASR (locally) or the default GPU kernel on Colab.

## 1. Config

In [ ]:
LANGUAGE = "ven"
BASE_MODEL = "facebook/wav2vec2-xls-r-300m"   # proposal primary; alt: facebook/wav2vec2-large-xlsr-53

# Stage 1: NCHLT + ANV combined pre-training
# Stage 2: NCHLT-only refinement, starting from the Stage 1 output
STAGE = 1

STAGE1_OUTPUT = f"./wav2vec2-{LANGUAGE}-stage1"
STAGE2_OUTPUT = f"./wav2vec2-{LANGUAGE}-stage2"

DATA = "../dataset/processed"
TOKENIZER_DIR = "../tokenizers/ven"

if STAGE == 1:
    TRAIN_FILES = [f"{DATA}/nchlt_ven/train.csv", f"{DATA}/anv_ven/train.csv"]
    EVAL_FILES = [f"{DATA}/nchlt_ven/validation.csv", f"{DATA}/anv_ven/dev.csv"]
    MODEL_SOURCE = BASE_MODEL
    OUTPUT_DIR = STAGE1_OUTPUT
else:
    TRAIN_FILES = [f"{DATA}/nchlt_ven/train.csv"]
    EVAL_FILES = [f"{DATA}/nchlt_ven/validation.csv"]
    MODEL_SOURCE = STAGE1_OUTPUT  # resume from stage 1
    OUTPUT_DIR = STAGE2_OUTPUT

PER_DEVICE_TRAIN_BATCH = 8      # drop to 4 (or 2) on CUDA OOM
PER_DEVICE_EVAL_BATCH = 8
GRAD_ACCUM_STEPS = 2
LEARNING_RATE = 1e-4
NUM_EPOCHS = 10                 # early stopping usually ends it sooner
EARLY_STOPPING_PATIENCE = 3

## 2. Imports

In [ ]:
import numpy as np
import torch
from dataclasses import dataclass
from typing import Dict, List, Union
from datasets import load_dataset, Audio, Features, Value
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from jiwer import wer, cer

## 3. Load data

datasets >= 5: the Audio feature must be declared at load time (cast_column on CSV string columns raises ArrowNotImplementedError).

In [ ]:
features = Features({"audio": Audio(sampling_rate=16000), "transcript": Value("string")})

dataset_dict = load_dataset(
    "csv",
    data_files={"train": TRAIN_FILES, "dev": EVAL_FILES},
    features=features,
)
dataset_dict

## 4. Processor (shared committed tokenizer)

In [ ]:
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(TOKENIZER_DIR)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=16000, padding_value=0.0,
    do_normalize=True, return_attention_mask=True,
)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
print("vocab size:", len(processor.tokenizer))

In [ ]:
def prepare_dataset(batch):
    samples = batch["audio"].get_all_samples()
    array = samples.data.numpy().squeeze()
    batch["input_values"] = processor(array, sampling_rate=16000).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor.tokenizer(batch["transcript"]).input_ids
    return batch

dataset_dict = dataset_dict.map(prepare_dataset, remove_columns=dataset_dict["train"].column_names)

## 5. Data collator + metrics

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.pad(labels=label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

In [ ]:
def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    return {"wer": wer(label_str, pred_str), "cer": cer(label_str, pred_str)}

## 6. Model

In [ ]:
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_SOURCE,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True,  # fresh CTC head sized to our vocab
)
model.freeze_feature_encoder()  # keep pretrained low-level audio features frozen
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

## 7. Training

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    num_train_epochs=NUM_EPOCHS,
    fp16=torch.cuda.is_available(),
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    push_to_hub=False,
    report_to=[],
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["dev"],
    processing_class=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

In [ ]:
# GPU required - do not run on the MacBook
trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"saved to {OUTPUT_DIR}")

## 8. Evaluate on the held-out test set

In [ ]:
test_features = Features({"audio": Audio(sampling_rate=16000), "transcript": Value("string")})
test_ds = load_dataset("csv", data_files={"test": f"{DATA}/nchlt_ven/test.csv"}, features=test_features)["test"]
test_ds = test_ds.map(prepare_dataset, remove_columns=test_ds.column_names)
print(trainer.evaluate(test_ds))